#Project FRM-GIS

## 1. Initialize Google Earth Engine

Load the Google Cloud Project ID from Colab Secrets, authenticate
with Google Earth Engine, and verify that the connection works.

In [1]:
%pip install -q geemap

In [2]:
!pip install earthengine-api geemap

In [4]:

import ee
from google.colab import userdata

PROJECT_ID = userdata.get("PROJECT_ID")

if not PROJECT_ID:
    raise ValueError(
        "PROJECT_ID not found. Check Colab Secrets and notebook access."
    )

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print(ee.String("Earth Engine connection successful").getInfo())


Earth Engine connection successful


In [5]:
# Verify that the SRTM 30 m elevation dataset is accessible
srtm = ee.Image("USGS/SRTMGL1_003")

print("Dataset:", "SRTM 30 m DEM")
print("Available bands:", srtm.bandNames().getInfo())
print("Dataset access verified.")

Dataset: SRTM 30 m DEM
Available bands: ['elevation']
Dataset access verified.


##2.1 Inspect candidate watershed boundaries

Use HydroBASINS Level 7 to identify candidate catchments around
the Pamba River Basin. Inspect the results before selecting an AOI.

In [6]:
import geemap

# Approximate search window around the Pamba Basin.
# This is only for finding candidate catchments, NOT the final AOI.
search_area = ee.Geometry.Rectangle([76.2, 8.5, 77.4, 10.2])

# Load candidate watershed polygons
basins = ee.FeatureCollection(
    "WWF/HydroSHEDS/v1/Basins/hybas_7"
).filterBounds(search_area)

print("Candidate basin polygons:", basins.size().getInfo())

# Inspect available attributes
print(
    "First candidate attributes:",
    basins.first().toDictionary().getInfo()
)

Candidate basin polygons: 14
First candidate attributes: {'COAST': 0, 'DIST_MAIN': 363.1, 'DIST_SINK': 363.1, 'ENDO': 0, 'HYBAS_ID': 4071150410, 'MAIN_BAS': 4070028760, 'NEXT_DOWN': 4071149830, 'NEXT_SINK': 4070028760, 'ORDER': 2, 'PFAF_ID': 4538029, 'SORT': 5322, 'SUB_AREA': 1543, 'UP_AREA': 1543.4}


In [7]:
Map = geemap.Map(
    basemap="Esri.WorldTopoMap"
)

Map.centerObject(search_area, 8)

Map.addLayer(
    search_area,
    {"color": "yellow"},
    "Search window"
)

Map.addLayer(
    basins.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS candidates"
)

Map

Map(center=[9.349811297242743, 76.79999999999993], controls=(WidgetControl(options=['position', 'transparent_b…

## 2.2 Inspect HydroBASINS Candidate Attributes

Check the number of candidate features and their available
properties before selecting the Pamba River Basin boundary.

In [8]:
# Count candidate basin features
print("Number of basin candidates:", basins.size().getInfo())

# Inspect available attributes
first_basin = ee.Feature(basins.first())

print("\nAvailable properties:")
print(first_basin.propertyNames().getInfo())

# Display a small sample of attributes without geometries
sample = (
    basins
    .limit(5)
    .select(first_basin.propertyNames(), None, False)
)

print("\nFirst five candidate records:")
print(sample.getInfo()["features"])

Number of basin candidates: 14

Available properties:
['SUB_AREA', 'COAST', 'PFAF_ID', 'DIST_MAIN', 'HYBAS_ID', 'DIST_SINK', 'NEXT_DOWN', 'ORDER', 'ENDO', 'MAIN_BAS', 'NEXT_SINK', 'SORT', 'system:index', 'UP_AREA']

First five candidate records:
[{'type': 'Feature', 'geometry': None, 'id': '0001000000000000052a', 'properties': {'COAST': 0, 'DIST_MAIN': 363.1, 'DIST_SINK': 363.1, 'ENDO': 0, 'HYBAS_ID': 4071150410, 'MAIN_BAS': 4070028760, 'NEXT_DOWN': 4071149830, 'NEXT_SINK': 4070028760, 'ORDER': 2, 'PFAF_ID': 4538029, 'SORT': 5322, 'SUB_AREA': 1543, 'UP_AREA': 1543.4}}, {'type': 'Feature', 'geometry': None, 'id': '0001000000000000070e', 'properties': {'COAST': 0, 'DIST_MAIN': 0, 'DIST_SINK': 0, 'ENDO': 0, 'HYBAS_ID': 4070029250, 'MAIN_BAS': 4070029250, 'NEXT_DOWN': 0, 'NEXT_SINK': 4070029250, 'ORDER': 1, 'PFAF_ID': 4539020, 'SORT': 5351, 'SUB_AREA': 6713.5, 'UP_AREA': 6714.3}}, {'type': 'Feature', 'geometry': None, 'id': '00010000000000000710', 'properties': {'COAST': 0, 'DIST_MAIN': 0,

## 2.3 Compare HydroBASINS Candidates

Create a compact attribute table for all candidate polygons
to support evidence-based selection of the Pamba River Basin.

In [9]:
import pandas as pd

fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "MAIN_BAS",
    "NEXT_DOWN",
    "DIST_MAIN",
    "ORDER"
]

candidate_features = (
    basins
    .select(fields, None, False)
    .getInfo()["features"]
)

candidate_df = pd.DataFrame(
    [feature["properties"] for feature in candidate_features]
)

candidate_df = candidate_df.sort_values(
    by="SUB_AREA",
    ascending=False
).reset_index(drop=True)

display(candidate_df.round(2))

,DIST_MAIN,HYBAS_ID,MAIN_BAS,NEXT_DOWN,ORDER,PFAF_ID,SUB_AREA,UP_AREA
0,0.0,4070029250,4070029250,0,1,4539020,6713.5,6714.3
1,0.0,4070029810,4070029810,0,1,4540200,6376.7,6376.9
2,0.0,4070029440,4070029440,0,1,4539080,5625.5,5625.7
3,0.0,4070029360,4070029360,0,1,4539060,5159.1,5159.1
4,0.0,4070029530,4070029530,0,0,4540101,4728.5,4728.5
5,0.0,4070029720,4070029720,0,1,4540104,3955.6,3955.6
6,0.0,4070029770,4070029770,0,1,4540108,1809.4,1809.5
7,363.1,4071150410,4070028760,4071149830,2,4538029,1543.0,1543.4
8,0.0,4070029680,4070029680,0,1,4540102,1507.1,1507.1
9,0.0,4070029750,4070029750,0,1,4540106,1318.6,1318.9


## 2.6 Geographic Verification of HydroBASINS Candidates

Calculate each candidate's centroid and geometric area to help
identify which polygons overlap the intended Pamba catchment.

In [10]:
def add_geometry_metrics(feature):
    centroid = feature.geometry().centroid(maxError=100)
    coords = centroid.coordinates()

    return feature.set({
        "centroid_lon": coords.get(0),
        "centroid_lat": coords.get(1),
        "geometry_area_km2": feature.geometry().area(maxError=100).divide(1e6)
    })

basins_with_metrics = basins.map(add_geometry_metrics)

metric_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "centroid_lon",
    "centroid_lat",
    "geometry_area_km2"
]

features = (
    basins_with_metrics
    .select(metric_fields, None, False)
    .getInfo()["features"]
)

geo_df = pd.DataFrame([
    feature["properties"] for feature in features
])

geo_df = geo_df.sort_values(
    "centroid_lat",
    ascending=False
).reset_index(drop=True)

display(geo_df.round(4))

,HYBAS_ID,PFAF_ID,SUB_AREA,UP_AREA,centroid_lat,centroid_lon,geometry_area_km2
0,4071150410,4538029,1543.0,1543.4,10.3886,77.2861,1542.9605
1,4070029810,4540200,6376.7,6376.9,10.0549,76.8511,6376.6909
2,4070029780,4540109,660.9,660.9,9.9732,76.3438,660.9421
3,4070029770,4540108,1809.4,1809.5,9.8943,76.6432,1809.3996
4,4070029250,4539020,6713.5,6714.3,9.8903,77.8653,6713.5544
5,4070029730,4540105,543.9,543.9,9.6691,76.3299,543.8686
6,4070029750,4540106,1318.6,1318.9,9.6245,76.6417,1318.6210
7,4070029760,4540107,87.5,87.5,9.5837,76.4453,87.4622
8,4070029360,4539060,5159.1,5159.1,9.3972,77.7572,5159.1107
9,4070029720,4540104,3955.6,3955.6,9.3345,76.8777,3955.6393


### 2.7 Screen HydroBASINS Candidates by Geographic Extent

Compare each candidate's bounding coordinates with the official
Pamba basin map to identify polygons worth inspecting visually.

This is a preliminary screening step, not final boundary validation.

In [11]:
def add_bbox_metrics(feature):
    ring = ee.List(
        feature.geometry()
        .bounds(maxError=100)
        .coordinates()
        .get(0)
    )

    lons = ring.map(lambda point: ee.List(point).get(0))
    lats = ring.map(lambda point: ee.List(point).get(1))

    return feature.set({
        "min_lon": ee.Number(lons.reduce(ee.Reducer.min())),
        "max_lon": ee.Number(lons.reduce(ee.Reducer.max())),
        "min_lat": ee.Number(lats.reduce(ee.Reducer.min())),
        "max_lat": ee.Number(lats.reduce(ee.Reducer.max()))
    })

basins_bbox = basins.map(add_bbox_metrics)

bbox_fields = [
    "HYBAS_ID", "PFAF_ID", "SUB_AREA",
    "min_lon", "max_lon", "min_lat", "max_lat"
]

bbox_features = (
    basins_bbox
    .select(bbox_fields, None, False)
    .getInfo()["features"]
)

bbox_df = pd.DataFrame([
    f["properties"] for f in bbox_features
])

display(
    bbox_df.sort_values("SUB_AREA")
    .reset_index(drop=True)
    .round(4)
)

,HYBAS_ID,PFAF_ID,SUB_AREA,max_lat,max_lon,min_lat,min_lon
0,4070029760,4540107,87.5,9.6375,76.5083,9.5208,76.3786
1,4070029690,4540103,502.9,9.3125,76.7875,8.9375,76.3792
2,4070029730,4540105,543.9,9.9750,76.4208,9.3167,76.2328
3,4070029780,4540109,660.9,10.1917,76.4795,9.6417,76.1625
4,4070029750,4540106,1318.6,9.8458,76.9208,9.3708,76.3828
5,4070029680,4540102,1507.1,9.1750,77.2625,8.7250,76.5333
6,4071150410,4538029,1543.0,10.7000,77.5083,10.1244,77.0619
7,4070029770,4540108,1809.4,10.1337,76.9417,9.6161,76.3994
8,4070029720,4540104,3955.6,9.6708,77.2833,9.0208,76.3625
9,4070029530,4540101,4728.5,9.0083,77.6500,8.0744,76.5333


## 2.7 Screen HydroBASINS Candidates by Geographic Extent

Compare each candidate's bounding coordinates with the official
Pamba basin map to identify polygons worth inspecting visually.

This is a preliminary screening step, not final boundary validation.

In [12]:
def add_bbox_metrics(feature):
    ring = ee.List(
        feature.geometry()
        .bounds(maxError=100)
        .coordinates()
        .get(0)
    )

    lons = ring.map(lambda point: ee.List(point).get(0))
    lats = ring.map(lambda point: ee.List(point).get(1))

    return feature.set({
        "min_lon": ee.Number(lons.reduce(ee.Reducer.min())),
        "max_lon": ee.Number(lons.reduce(ee.Reducer.max())),
        "min_lat": ee.Number(lats.reduce(ee.Reducer.min())),
        "max_lat": ee.Number(lats.reduce(ee.Reducer.max()))
    })

basins_bbox = basins.map(add_bbox_metrics)

bbox_fields = [
    "HYBAS_ID", "PFAF_ID", "SUB_AREA",
    "min_lon", "max_lon", "min_lat", "max_lat"
]

bbox_features = (
    basins_bbox
    .select(bbox_fields, None, False)
    .getInfo()["features"]
)

bbox_df = pd.DataFrame([
    f["properties"] for f in bbox_features
])

display(
    bbox_df.sort_values("SUB_AREA")
    .reset_index(drop=True)
    .round(4)
)

,HYBAS_ID,PFAF_ID,SUB_AREA,max_lat,max_lon,min_lat,min_lon
0,4070029760,4540107,87.5,9.6375,76.5083,9.5208,76.3786
1,4070029690,4540103,502.9,9.3125,76.7875,8.9375,76.3792
2,4070029730,4540105,543.9,9.9750,76.4208,9.3167,76.2328
3,4070029780,4540109,660.9,10.1917,76.4795,9.6417,76.1625
4,4070029750,4540106,1318.6,9.8458,76.9208,9.3708,76.3828
5,4070029680,4540102,1507.1,9.1750,77.2625,8.7250,76.5333
6,4071150410,4538029,1543.0,10.7000,77.5083,10.1244,77.0619
7,4070029770,4540108,1809.4,10.1337,76.9417,9.6161,76.3994
8,4070029720,4540104,3955.6,9.6708,77.2833,9.0208,76.3625
9,4070029530,4540101,4728.5,9.0083,77.6500,8.0744,76.5333


## 2.8 Verify the HydroBASINS Dataset Level

Confirm the source collection and its metadata before selecting a catchment.

The current candidates have not yet been matched to the official Pamba River Basin boundary. Dataset-level verification helps avoid selecting polygons from an unintended HydroBASINS layer.

In [13]:
# Inspect the collection source and candidate identifiers.
# This does not change or select any basin.

print("Candidate count:", basins.size().getInfo())

print("\nCandidate HYBAS_ID values:")
print(
    basins.aggregate_array("HYBAS_ID")
    .getInfo()
)

print("\nCandidate PFAF_ID values:")
print(
    basins.aggregate_array("PFAF_ID")
    .getInfo()
)

Candidate count: 14

Candidate HYBAS_ID values:
[4071150410, 4070029250, 4070029360, 4070029440, 4070029680, 4070029720, 4070029750, 4070029770, 4070029810, 4070029530, 4070029690, 4070029730, 4070029760, 4070029780]

Candidate PFAF_ID values:
[4538029, 4539020, 4539060, 4539080, 4540102, 4540104, 4540106, 4540108, 4540200, 4540101, 4540103, 4540105, 4540107, 4540109]


## 2.9 Compare HydroBASINS Levels

Compare Levels 6, 7, and 8 within the existing search area. For each level, inspect the five candidate catchments whose reported sub-basin areas are closest to the published Pamba basin area of approximately 2,235 km².

Area similarity is used only to narrow the candidates. Final selection requires geographic and boundary verification.

In [14]:
REFERENCE_AREA_KM2 = 2235

level_results = []

for level in [6, 7, 8]:
    collection_id = (
        f"WWF/HydroSHEDS/v1/Basins/hybas_{level}"
    )

    candidates = (
        ee.FeatureCollection(collection_id)
        .filterBounds(search_area)
    )

    def add_comparison_metrics(feature):
        centroid = feature.geometry().centroid(
            maxError=100
        ).coordinates()

        area = ee.Number(feature.get("SUB_AREA"))

        return feature.set({
            "level": level,
            "area_difference_km2": area.subtract(
                REFERENCE_AREA_KM2
            ).abs(),
            "centroid_lon": centroid.get(0),
            "centroid_lat": centroid.get(1)
        })

    closest = (
        candidates
        .map(add_comparison_metrics)
        .sort("area_difference_km2")
        .limit(5)
        .select([
            "level",
            "HYBAS_ID",
            "PFAF_ID",
            "SUB_AREA",
            "UP_AREA",
            "area_difference_km2",
            "centroid_lon",
            "centroid_lat"
        ])
    )

    result = closest.getInfo()

    for feature in result["features"]:
        level_results.append(feature["properties"])

    print(
        f"Level {level}: "
        f"{candidates.size().getInfo()} candidates "
        "intersect the search area"
    )

level_df = pd.DataFrame(level_results)

display(
    level_df.sort_values(
        ["level", "area_difference_km2"]
    ).round(3).reset_index(drop=True)
)

Level 6: 6 candidates intersect the search area
Level 7: 14 candidates intersect the search area
Level 8: 45 candidates intersect the search area


,HYBAS_ID,PFAF_ID,SUB_AREA,UP_AREA,area_difference_km2,centroid_lat,centroid_lon,level
0,4060029360,453906,5159.1,5159.1,2924.1,9.397,77.757,6
1,4060029440,453908,5625.5,5625.7,3390.5,8.824,77.588,6
2,4060029810,454020,6376.7,6376.9,4141.7,10.055,76.851,6
3,4060029250,453902,6713.5,6714.3,4478.5,9.890,77.865,6
4,4061148830,453802,9054.8,9054.8,6819.8,10.590,77.628,6
5,4070029770,4540108,1809.4,1809.5,425.6,9.894,76.643,7
6,4071150410,4538029,1543.0,1543.4,692.0,10.389,77.286,7
7,4070029680,4540102,1507.1,1507.1,727.9,8.976,76.946,7
8,4070029750,4540106,1318.6,1318.9,916.4,9.625,76.642,7
9,4070029780,4540109,660.9,660.9,1574.1,9.973,76.344,7


## 2.10 Visually Screen Level 8 Catchment Boundaries

Display Level 8 HydroBASINS polygons intersecting the approximate
geographic extent of the official Pamba River Basin map.

The rectangular extent is for screening only. It is not the basin
boundary, and no candidate will be selected from this overlay alone.

In [15]:
# Approximate extent read from the official basin map.
# This rectangle is NOT the Pamba basin boundary.

pamba_map_extent = ee.Geometry.Rectangle([
    76.20, 9.10, 77.25, 9.80
])

level8_candidates = (
    ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8")
    .filterBounds(pamba_map_extent)
)

print(
    "Level 8 polygons intersecting the approximate map extent:",
    level8_candidates.size().getInfo()
)

Map.centerObject(pamba_map_extent, 9)

Map.addLayer(
    pamba_map_extent,
    {"color": "yellow"},
    "Approximate official-map extent"
)

Map.addLayer(
    level8_candidates.style(
        color="cyan",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS Level 8 candidates"
)

Map

Level 8 polygons intersecting the approximate map extent: 20


Map(bottom=31058.0, center=[9.45027091655913, 76.72500000000021], controls=(WidgetControl(options=['position',…

## 2.11 Label HydroBASINS Candidates for Identification

Display the Level 8 candidate catchments with their identifiers and
reported sub-basin areas to connect the map view with the attribute table.

Use the labels to inspect spatial relationships. Do not treat partial
overlap with the official Pamba map as boundary confirmation.

In [16]:
# Prepare concise labels for the candidate polygons.
def add_candidate_label(feature):
    label = ee.String(
        ee.Number(feature.get("HYBAS_ID")).format()
    ).cat(" | ").cat(
        ee.Number(feature.get("SUB_AREA")).format("%.0f")
    ).cat(" km²")

    return feature.set("map_label", label)

labeled_level8 = level8_candidates.map(add_candidate_label)

# Add candidate outlines and labels.
Map.addLayer(
    labeled_level8.style(
        color="cyan",
        fillColor="00000000",
        width=2
    ),
    {},
    "Level 8 candidate boundaries"
)

Map.addLayer(
    labeled_level8.style(
        color="white",
        pointSize=1
    ),
    {},
    "Candidate label anchors"
)

Map

Map(bottom=62080.0, center=[9.45027091655913, 76.72500000000021], controls=(WidgetControl(options=['position',…

## 2.12 Tabulate Level 8 Candidates Within the Pamba Map Extent

List the Level 8 catchments intersecting the approximate extent of the
official Pamba basin map, including their IDs, areas, and centroids.

Use this table to narrow the candidates for further geographic
verification. Do not select a basin solely by area or centroid.

In [17]:
# Reuse pamba_map_extent and level8_candidates from Section 2.10.

def candidate_summary(feature):
    centroid = feature.geometry().centroid(
        maxError=100
    ).coordinates()

    return feature.set({
        "centroid_lon": centroid.get(0),
        "centroid_lat": centroid.get(1),
        "geometry_area_km2": (
            feature.geometry().area(maxError=100).divide(1e6)
        )
    })

summary_fc = level8_candidates.map(candidate_summary)

summary_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "centroid_lat",
    "centroid_lon",
    "geometry_area_km2"
]

summary_info = (
    summary_fc
    .select(summary_fields, None, False)
    .getInfo()
)

summary_df = pd.DataFrame([
    feature["properties"]
    for feature in summary_info["features"]
])

summary_df = summary_df.sort_values(
    "centroid_lat"
).reset_index(drop=True)

print("Level 8 candidates:", len(summary_df))
display(summary_df.round(4))

Level 8 candidates: 20


,HYBAS_ID,PFAF_ID,SUB_AREA,UP_AREA,centroid_lat,centroid_lon,geometry_area_km2
0,4080029680,45401020,1507.1,1507.1,8.9762,76.9461,1507.1544
1,4081154940,45390809,397.3,397.3,9.0697,77.3201,397.2688
2,4080029690,45401030,502.9,502.9,9.1276,76.5589,502.8656
3,4081153890,45401042,1135.9,1136.2,9.1876,76.8871,1135.8836
4,4081153960,45401046,274.3,274.3,9.2494,77.0007,274.2778
5,4080029720,45401041,320.9,3955.4,9.3307,76.4682,320.8646
6,4081153820,45401048,314.2,314.3,9.3361,76.9992,314.2289
7,4081153830,45401045,299.3,1680.2,9.3473,76.7345,299.2986
8,4081153910,45401047,9.3,1106.8,9.3586,76.8376,9.3482
9,4081153850,45401043,16.3,2498.7,9.3671,76.5823,16.3192


## 2.13 Inspect HydroBASINS Drainage Connectivity

Update the approximate screening extent using the coordinate range
shown on the official Pamba River Basin map.

Inspect Level 8 catchments together with their downstream links,
main-basin identifiers, reported areas, and geographic positions.

The purpose is to understand how candidate catchments connect
hydrologically before attempting to identify or assemble the basin boundary.

In [18]:
# Broad screening extent based on the official map's coordinate grid.
# This rectangle is NOT the Pamba basin boundary.

pamba_screening_extent = ee.Geometry.Rectangle([
    76.20, 8.85,
    77.25, 9.70
])

level8_screen = (
    ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8")
    .filterBounds(pamba_screening_extent)
)

def add_candidate_metrics(feature):
    centroid = feature.geometry().centroid(
        maxError=100
    ).coordinates()

    geometry_area = (
        feature.geometry()
        .area(maxError=100)
        .divide(1e6)
    )

    return feature.set({
        "centroid_lon": centroid.get(0),
        "centroid_lat": centroid.get(1),
        "geometry_area_km2": geometry_area
    })

level8_screen_metrics = level8_screen.map(
    add_candidate_metrics
)

connectivity_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "NEXT_DOWN",
    "MAIN_BAS",
    "NEXT_SINK",
    "COAST",
    "ORDER",
    "DIST_MAIN",
    "DIST_SINK",
    "centroid_lat",
    "centroid_lon",
    "geometry_area_km2"
]

connectivity_info = (
    level8_screen_metrics
    .select(connectivity_fields, None, False)
    .getInfo()
)

connectivity_df = pd.DataFrame([
    feature["properties"]
    for feature in connectivity_info["features"]
])

connectivity_df = connectivity_df.sort_values(
    ["centroid_lat", "centroid_lon"]
).reset_index(drop=True)

print(
    "Level 8 candidates in updated screening extent:",
    len(connectivity_df)
)

display(connectivity_df.round(4))

Level 8 candidates in updated screening extent: 23


,COAST,DIST_MAIN,DIST_SINK,HYBAS_ID,MAIN_BAS,NEXT_DOWN,NEXT_SINK,ORDER,PFAF_ID,SUB_AREA,UP_AREA,centroid_lat,centroid_lon,geometry_area_km2
0,0,0.0,0.0,4080029660,4080029660,0,4080029660,1,45401018,632.5,632.5,8.8896,76.8287,632.4776
1,1,0.0,0.0,4080029670,4080029670,0,4080029670,0,45401019,133.7,133.7,8.8987,76.6177,133.6722
2,0,110.2,110.2,4081154950,4080029440,4081155070,4080029440,2,45390808,271.3,271.4,8.9598,77.2665,271.3256
3,0,0.0,0.0,4080029680,4080029680,0,4080029680,1,45401020,1507.1,1507.1,8.9762,76.9461,1507.1544
4,0,110.2,110.2,4081154940,4080029440,4081155070,4080029440,1,45390809,397.3,397.3,9.0697,77.3201,397.2688
5,1,0.0,0.0,4080029690,4080029690,0,4080029690,0,45401030,502.9,502.9,9.1276,76.5589,502.8656
6,0,24.8,24.8,4081153890,4080029720,4080029720,4080029720,2,45401042,1135.9,1136.2,9.1876,76.8871,1135.8836
7,0,66.6,66.6,4081153960,4080029720,4081153830,4080029720,2,45401046,274.3,274.3,9.2494,77.0007,274.2778
8,0,0.0,0.0,4080029720,4080029720,0,4080029720,1,45401041,320.9,3955.4,9.3307,76.4682,320.8646
9,0,70.1,70.1,4081153820,4080029720,4081153910,4080029720,2,45401048,314.2,314.3,9.3361,76.9992,314.2289


## 2.14 Inspect an Official Hydrological Boundary Dataset

Load the downloaded government sub-basin boundary archive into Colab
and inspect its contents before processing any geometry.

The dataset must be checked for its source, coordinate reference system,
attribute fields, and geographic coverage before use.

In [19]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_files = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_files:
    raise ValueError(
        "Please upload the downloaded boundary dataset as a ZIP file."
    )

boundary_zip = zip_files[0]
extract_dir = "/content/official_boundary_data"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(boundary_zip, "r") as archive:
    archive.extractall(extract_dir)
    archive_contents = archive.namelist()

print("Archive:", boundary_zip)
print("Extracted files:")

for name in archive_contents:
    print("-", name)

Saving Subbasin.zip to Subbasin.zip
Archive: Subbasin.zip
Extracted files:
- Subbasin.shx
- Subbasin.cpg
- Subbasin.dbf
- Subbasin.prj
- Subbasin.sbn
- Subbasin.sbx
- Subbasin.shp
- Subbasin.shp.xml


## 2.15 Load and Inspect Official Sub-Basin Boundaries

Load the extracted shapefile with GeoPandas and inspect its coordinate reference system, attribute fields, feature count, and sample records. These checks will help us identify sub-basins using their attributes and geographic locations before selecting the Pamba River Basin boundary.


In [20]:
import geopandas as gpd
import os

shapefile_path = os.path.join(
    extract_dir,
    "Subbasin.shp"
)

if not os.path.exists(shapefile_path):
    raise FileNotFoundError(
        f"Shapefile not found at: {shapefile_path}"
    )

subbasins = gpd.read_file(shapefile_path)

print("Feature count:", len(subbasins))
print("CRS:", subbasins.crs)
print("\nAttribute columns:")
print(subbasins.columns.tolist())

print("\nFirst five records:")
display(subbasins.drop(columns="geometry").head())

print("\nGeometry types:")
print(subbasins.geometry.geom_type.value_counts())

print("\nTotal bounds (minx, miny, maxx, maxy):")
print(subbasins.total_bounds)


Feature count: 101
CRS: PROJCS["WGS_1984_Lambert_Conformal_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",24],PARAMETER["central_meridian",80],PARAMETER["standard_parallel_1",12.4729444],PARAMETER["standard_parallel_2",35.17280555],PARAMETER["false_easting",4000000],PARAMETER["false_northing",4000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Attribute columns:
['objectid', 'sbconc', 'sbcode', 'bacode', 'ba_name', 'sub_basin', 'shape_Leng', 'shape_Area', 'geometry']

First five records:


,objectid,sbconc,sbcode,bacode,ba_name,sub_basin,shape_Leng,shape_Area
0,1,C2AARA,ARA,2A,Ganga Basin,Above Ramganga Confluence,2.338072e+06,3.910461e+10
1,2,C07BAI,BAI,07,Brahmani and Baitarni Basin,Baitarni,9.143105e+05,1.435123e+10
2,3,A01BAM,BAM,01,Indus (Up to border) Basin,Barmer,1.241011e+06,2.165894e+10
3,4,C2ABAN,BAN,2A,Ganga Basin,Banas,2.054371e+06,5.165151e+10
4,5,D2CBAR,BAR,2C,Barak and others Basin,Barak,1.440719e+06,2.765898e+10



Geometry types:
Polygon         86
MultiPolygon    15
Name: count, dtype: int64

Total bounds (minx, miny, maxx, maxy):
[2819242.6951 2177948.4985 5679303.797  5443428.0539]


## 2.16 Search Official Sub-Basin Names

Inspect all basin and sub-basin names, then filter for records containing likely matches for Pamba, Kerala, and nearby regional names. This is an initial text-based search; absence of a direct “Pamba” match does not mean the dataset lacks the relevant boundary.


In [21]:
# Convert a copy to WGS84 for geographic inspection
subbasins_wgs84 = subbasins.to_crs(epsg=4326)

# Display all official basin and sub-basin names
name_table = (
    subbasins_wgs84[
        ["objectid", "sbcode", "bacode", "ba_name", "sub_basin"]
    ]
    .sort_values(["ba_name", "sub_basin"])
    .reset_index(drop=True)
)

display(name_table)

# Search likely names without assuming Pamba is explicitly listed
keywords = r"pamba|pampa|kerala|periyar|bharathapuzha|west flowing"

matches = name_table[
    name_table["ba_name"].str.contains(
        keywords, case=False, na=False, regex=True
    )
    | name_table["sub_basin"].str.contains(
        keywords, case=False, na=False, regex=True
    )
]

print("\nPotentially relevant records:")
display(matches)

print("\nGeographic bounds by candidate (longitude/latitude):")
if not matches.empty:
    bounds = subbasins_wgs84.loc[
        matches.index, ["objectid", "sub_basin", "geometry"]
    ].copy()
    bounds[["min_lon", "min_lat", "max_lon", "max_lat"]] = (
        bounds.geometry.bounds
    )
    display(bounds.drop(columns="geometry"))


,objectid,sbcode,bacode,ba_name,sub_basin
0,76,SHK,23,Area of North Ladakha not draining into Indus ...,Shaksgam
1,81,SUM,23,Area of North Ladakha not draining into Indus ...,Sulmar
2,5,BAR,2C,Barak and others Basin,Barak
3,46,KYN,2C,Barak and others Basin,Kynchiang and other south flowing rivers
4,59,NAO,2C,Barak and others Basin,Naoch chara and others
...,...,...,...,...,...
96,70,RAN,20,West flowing rivers of Kutch and Saurashtra in...,Drainage of Rann
97,48,LUL,20,West flowing rivers of Kutch and Saurashtra in...,Luni Lower
98,49,LUU,20,West flowing rivers of Kutch and Saurashtra in...,Luni Upper
99,73,SAR,20,West flowing rivers of Kutch and Saurashtra in...,Saraswati



Potentially relevant records:


,objectid,sbcode,bacode,ba_name,sub_basin
14,63,PAM,19,East flowing rivers South of Cauvery Basin,Pamba and others
90,10,BHT,14,West flowing rivers South of Tapi Basin,Bhatsol and others
91,61,NET,14,West flowing rivers South of Tapi Basin,Netravati and others
92,64,PAR,14,West flowing rivers South of Tapi Basin,Periyar and others
93,92,VAR,14,West flowing rivers South of Tapi Basin,Varrar and others
94,93,VAS,14,West flowing rivers South of Tapi Basin,Vasishti and others
95,7,BHA,20,West flowing rivers of Kutch and Saurashtra in...,Bhadar and other west flowing rivers
96,70,RAN,20,West flowing rivers of Kutch and Saurashtra in...,Drainage of Rann
97,48,LUL,20,West flowing rivers of Kutch and Saurashtra in...,Luni Lower
98,49,LUU,20,West flowing rivers of Kutch and Saurashtra in...,Luni Upper



Geographic bounds by candidate (longitude/latitude):


,objectid,sub_basin,min_lon,min_lat,max_lon,max_lat
14,15,Cauvery Lower,78.030544,10.271314,79.879337,11.434858
90,91,Vamsadhara and other,83.327543,18.260317,85.230335,20.326161
91,92,Varrar and others,75.530021,10.311682,77.208478,11.815001
92,93,Vasishti and others,73.028110,14.262931,75.199246,17.999759
93,94,Wardha,75.971346,19.271944,79.782849,21.847867
94,95,Weinganga,78.032831,19.625290,80.879572,22.690954
95,96,Yamuna Lower,77.097817,22.869064,81.881688,28.029457
96,97,Yamuna Middle,75.814094,26.282900,79.336867,28.703405
97,98,Yamuna Upper,75.750409,27.311403,78.624310,31.421616
98,99,Mahi Lower,72.502085,21.671457,74.281089,23.650258


## 2.17 Verify the Official Pamba Sub-Basin Geometry

Select the official feature using its unique `sbcode` and inspect its attributes, geographic bounds, area, and geometry validity. This verifies that the correct feature was selected before using it as the study-area boundary.

In [22]:
# Select the Pamba sub-basin using its official code
pamba_official = subbasins_wgs84[
    subbasins_wgs84["sbcode"].astype(str).str.upper() == "PAM"
].copy()

if pamba_official.empty:
    raise ValueError("No feature with sbcode='PAM' was found.")

# Report the identifying attributes
display(
    pamba_official[
        ["objectid", "sbconc", "sbcode", "bacode",
         "ba_name", "sub_basin", "shape_Area"]
    ]
)

# Inspect geometry and geographic extent
print("Feature count:", len(pamba_official))
print("CRS:", pamba_official.crs)
print("Geometry type:", pamba_official.geometry.geom_type.tolist())
print("Geometry valid:", pamba_official.geometry.is_valid.tolist())
print("Bounds (min longitude, min latitude, max longitude, max latitude):")
print(pamba_official.total_bounds)

# Calculate area after projecting to the dataset's original metric CRS
pamba_metric = pamba_official.to_crs(subbasins.crs)
area_km2 = pamba_metric.geometry.area.sum() / 1_000_000

print(f"Calculated geometry area: {area_km2:,.1f} km²")


,objectid,sbconc,sbcode,bacode,ba_name,sub_basin,shape_Area
62,63,C19PAM,PAM,19,East flowing rivers South of Cauvery Basin,Pamba and others,1.828941e+10


Feature count: 1
CRS: EPSG:4326
Geometry type: ['Polygon']
Geometry valid: [False]
Bounds (min longitude, min latitude, max longitude, max latitude):
[77.13574639  9.208188   79.41615472 10.75229697]
Calculated geometry area: 18,289.4 km²


## 2.18 Diagnose the Official Boundary Geometry

Check the invalid geometry reason, the number of polygon components, and the feature's area in its original projected CRS. Also inspect the other sub-basins belonging to the same larger basin. This helps determine whether the record represents a broad regional grouping rather than the Pamba catchment itself.


In [23]:
from shapely.validation import explain_validity

feature = pamba_official.iloc[0]
geom = feature.geometry

print("Official sub-basin:", feature["sub_basin"])
print("Parent basin:", feature["ba_name"])
print("Validity reason:", explain_validity(geom))
print("Geometry type:", geom.geom_type)
print("Number of polygon components:",
      len(geom.geoms) if geom.geom_type == "MultiPolygon" else 1)
print("Geographic bounds:", geom.bounds)

# Inspect all records assigned to the same parent basin
parent_name = feature["ba_name"]

same_parent = subbasins_wgs84[
    subbasins_wgs84["ba_name"] == parent_name
].copy()

same_parent["area_km2"] = (
    same_parent.to_crs(subbasins.crs).geometry.area / 1_000_000
)

print("\nFeatures in the same parent basin:")
display(
    same_parent[
        ["objectid", "sbcode", "sub_basin", "area_km2"]
    ].sort_values("area_km2", ascending=False)
)

Official sub-basin: Pamba and others
Parent basin: East flowing rivers South of Cauvery Basin
Validity reason: Ring Self-intersection[78.2300539869383 9.84310665451539]
Geometry type: Polygon
Number of polygon components: 1
Geographic bounds: (77.1357463893732, 9.208188000589967, 79.41615471514856, 10.75229696726844)

Features in the same parent basin:


,objectid,sbcode,sub_basin,area_km2
89,90,VAI,Vaippar and others,20356.688642
62,63,PAM,Pamba and others,18289.413502


## 2.19 Narrow HydroBASINS Candidates to the Pamba Region

Filter Level 8 HydroBASINS polygons intersecting a tighter geographic window around the expected Pamba basin. Print their identifiers, sub-basin areas, upstream areas, and geographic bounds. These candidates will be reviewed spatially before selecting a basin or delineating a catchment from a DEM.

In [24]:

import ee
import pandas as pd

# Approximate Pamba-region screening window only — not the basin boundary
pamba_screening_extent = ee.Geometry.Rectangle(
    [76.20, 8.85, 77.25, 9.70]
)

# Reuse the existing Level 8 HydroBASINS collection: basins
# Filter polygons intersecting the screening window
regional_candidates = basins.filterBounds(pamba_screening_extent)

candidate_info = regional_candidates.map(
    lambda f: f.set(
        "centroid",
        f.geometry().centroid(100).coordinates()
    )
)

features = candidate_info.getInfo()["features"]

rows = []
for item in features:
    p = item["properties"]
    centroid = p.get("centroid", [None, None])
    geometry = item.get("geometry", {})
    coords = geometry.get("coordinates", [])

    rows.append({
        "HYBAS_ID": p.get("HYBAS_ID"),
        "PFAF_ID": p.get("PFAF_ID"),
        "SUB_AREA_km2": p.get("SUB_AREA"),
        "UP_AREA_km2": p.get("UP_AREA"),
        "COAST": p.get("COAST"),
        "ORDER": p.get("ORDER"),
        "centroid_lon": centroid[0] if len(centroid) > 0 else None,
        "centroid_lat": centroid[1] if len(centroid) > 1 else None,
    })

candidate_table = pd.DataFrame(rows)

if candidate_table.empty:
    print("No Level 8 candidates intersect this window.")
else:
    candidate_table = candidate_table.sort_values(
        ["SUB_AREA_km2", "UP_AREA_km2"],
        ascending=False
    ).reset_index(drop=True)

    print("Candidates intersecting screening window:",
          len(candidate_table))
    display(candidate_table)

Candidates intersecting screening window: 12


,HYBAS_ID,PFAF_ID,SUB_AREA_km2,UP_AREA_km2,COAST,ORDER,centroid_lon,centroid_lat
0,4070029250,4539020,6713.5,6714.3,0,1,77.865274,9.890266
1,4070029810,4540200,6376.7,6376.9,0,1,76.851059,10.054907
2,4070029440,4539080,5625.5,5625.7,0,1,77.587867,8.824237
3,4070029530,4540101,4728.5,4728.5,1,0,77.107027,8.537443
4,4070029720,4540104,3955.6,3955.6,0,1,76.877720,9.334498
5,4070029770,4540108,1809.4,1809.5,0,1,76.643221,9.894298
6,4070029680,4540102,1507.1,1507.1,0,1,76.946144,8.976178
7,4070029750,4540106,1318.6,1318.9,0,1,76.641748,9.624505
8,4070029780,4540109,660.9,660.9,1,0,76.343767,9.973194
9,4070029730,4540105,543.9,543.9,1,0,76.329946,9.669118


## 2.20 Visualize HydroBASINS Candidates

Display the 12 Level 8 polygons intersecting the Pamba-region screening window. Label each polygon with its HydroBASINS identifier and sub-area. The map is for visual screening only; the rectangular search window is not the study boundary.


In [28]:
import geemap

# Add labels and display properties to the candidate features
def add_candidate_label(feature):
    label = (
        ee.Number(feature.get("HYBAS_ID")).format()
        .cat(" | ")
        .cat(ee.Number(feature.get("SUB_AREA")).format("%.0f"))
        .cat(" km²")
    )
    return feature.set("map_label", label)

labeled_candidates = regional_candidates.map(add_candidate_label)

# Create a fresh map
candidate_map = geemap.Map(basemap="Esri.WorldTopoMap")
candidate_map.centerObject(pamba_screening_extent, 8)

candidate_map.addLayer(
    pamba_screening_extent,
    {"color": "yellow"},
    "Screening window"
)

candidate_map.addLayer(
    labeled_candidates.style(
        color="00FFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS candidate boundaries"
)

# Add centroid points to help associate map locations with the table
candidate_centroids = labeled_candidates.map(
    lambda f: ee.Feature(
        f.geometry().centroid(100),
        {
            "HYBAS_ID": f.get("HYBAS_ID"),
            "SUB_AREA": f.get("SUB_AREA")
        }
    )
)

candidate_map.addLayer(
    candidate_centroids,
    {"color": "red", "pointSize": 5},
    "Candidate centroids"
)

candidate_map

Map(center=[9.275210956308635, 76.72500000000022], controls=(WidgetControl(options=['position', 'transparent_b…

## 2.21 Inspect Candidate Drainage Units and Their Outlets

Inspect the HydroBASINS polygons individually and compare their locations with the Pamba River and its downstream outlet. This is a preliminary step to help identify a plausible outlet location before DEM-based watershed delineation.


In [ ]:
# Display the candidate table ordered by location and area
display(
    candidate_table[
        [
            "HYBAS_ID",
            "SUB_AREA_km2",
            "UP_AREA_km2",
            "COAST",
            "ORDER",
            "centroid_lon",
            "centroid_lat",
        ]
    ].sort_values(["centroid_lat", "centroid_lon"])
)

print(
    "\nReference area range: approximately 2,235–2,487 km²."
)
print(
    "Reminder: candidate area similarity alone does not establish "
    "that a polygon represents the Pamba catchment."
)

## 2.25 Load and Validate the Initial Watershed Delineation

Load the GeoJSON exported from the watershed tool. Separate the watershed polygon from the river-network features, verify geometry validity and geographic extent, and independently calculate the watershed area in a suitable projected coordinate system.


In [25]:
import geopandas as gpd
import pandas as pd
from google.colab import files

uploaded = files.upload()

geojson_files = [
    name for name in uploaded
    if name.lower().endswith((".geojson", ".json"))
]

if not geojson_files:
    raise ValueError("Upload the exported watershed GeoJSON file.")

watershed_file = geojson_files[0]
delineation = gpd.read_file(watershed_file)

print("File:", watershed_file)
print("CRS:", delineation.crs)
print("Total features:", len(delineation))
print("\nGeometry types:")
print(delineation.geometry.geom_type.value_counts())

# Separate polygon features from river-network lines
watershed_polygon = delineation[
    delineation.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

river_lines = delineation[
    delineation.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()

if watershed_polygon.empty:
    raise ValueError("No watershed polygon found in the GeoJSON.")

# Calculate area using a global equal-area CRS
watershed_equal_area = watershed_polygon.to_crs(epsg=6933)
calculated_area_km2 = (
    watershed_equal_area.geometry.area.sum() / 1_000_000
)

print("\nWatershed polygon count:", len(watershed_polygon))
print("River-network feature count:", len(river_lines))
print("Geometry valid:", watershed_polygon.geometry.is_valid.tolist())
print("Bounds (min lon, min lat, max lon, max lat):")
print(watershed_polygon.to_crs(epsg=4326).total_bounds)
print(f"Calculated watershed area: {calculated_area_km2:,.1f} km²")

Saving watershed (1).geojson to watershed (1).geojson
File: watershed (1).geojson
CRS: EPSG:4326
Total features: 86

Geometry types:
LineString      85
MultiPolygon     1
Name: count, dtype: int64

Watershed polygon count: 1
River-network feature count: 85
Geometry valid: [True]
Bounds (min lon, min lat, max lon, max lat):
[76.42875     9.01958333 77.28375     9.66625   ]
Calculated watershed area: 3,744.5 km²



## 2.26 Visually Inspect the Delineated Watershed

Overlay the exported watershed boundary, its river-network lines, and the selected outlet location on a topographic basemap. Inspect whether the outlet connects to the main river and whether the boundary appears to include neighbouring catchments.


In [26]:

import geemap
import ee

# Selected point used in the watershed tool: longitude, latitude
outlet = ee.Geometry.Point([76.4468, 9.3418])

# Create the inspection map
check_map = geemap.Map(basemap="Esri.WorldTopoMap")
check_map.centerObject(outlet, 9)

# Add the exported watershed polygon
check_map.add_gdf(
    watershed_polygon.to_crs(epsg=4326),
    layer_name="Delineated watershed",
    style={"color": "red", "fillColor": "00000000", "weight": 3}
)

# Add the river-network lines from the exported GeoJSON
if not river_lines.empty:
    check_map.add_gdf(
        river_lines.to_crs(epsg=4326),
        layer_name="Exported river network",
        style={"color": "blue", "weight": 2}
    )

# Add the selected outlet marker
check_map.addLayer(
    outlet,
    {"color": "yellow"},
    "Selected outlet point"
)

check_map

Map(center=[9.3418, 76.4468], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

## Step 2.30 — Inspect finer HydroBASINS candidates

**Objective:** Identify smaller hydrological catchments around the Pamba River Basin and compare their areas and locations with the official Kerala basin map.

**Method:** Query HydroBASINS level 9, filter catchments intersecting the Pamba search region, and calculate each candidate's approximate area and centroid.

**Important limitation:** A candidate's area being close to a published Pamba basin area does not prove that its boundary is correct. We will inspect its shape and river connections before accepting it.

In [27]:
import ee
import geemap
import pandas as pd
from IPython.display import display

# Broad search window around the Pamba basin.
# This is a search extent, NOT the final basin boundary.
pamba_search_area = ee.Geometry.Rectangle(
    [76.15, 9.00, 77.45, 9.85]
)

# HydroBASINS level 9: finer subdivisions than the level 8
# candidates examined earlier.
basins9 = ee.FeatureCollection(
    "WWF/HydroSHEDS/v1/Basins/hybas_9"
)

candidates9 = basins9.filterBounds(pamba_search_area)

# Add approximate area and centroid coordinates.
def add_candidate_metrics(feature):
    centroid = feature.geometry().centroid(1000).coordinates()
    area_km2 = feature.geometry().area(1000).divide(1_000_000)

    return feature.set({
        "area_km2": area_km2,
        "centroid_lonlat": centroid
    })

candidates9_metrics = candidates9.map(add_candidate_metrics)

# Retrieve only attributes, not full geometries.
candidate_data = candidates9_metrics.select([
    "HYBAS_ID",
    "NEXT_DOWN",
    "MAIN_BAS",
    "UP_AREA",
    "SUB_AREA",
    "area_km2",
    "centroid_lonlat"
]).getInfo()

rows = [
    feature["properties"]
    for feature in candidate_data["features"]
]

candidates_df = pd.DataFrame(rows)

if candidates_df.empty:
    print("No level-9 catchments intersect the search extent.")
else:
    candidates_df = candidates_df.sort_values(
        "area_km2", ascending=False
    ).reset_index(drop=True)

    print("Level-9 candidate count:", len(candidates_df))
    display(candidates_df)

    # Map the search region and candidate catchments.
    candidate_map = geemap.Map(basemap="Esri.WorldTopoMap")
    candidate_map.centerObject(pamba_search_area, 9)

    candidate_map.addLayer(
        pamba_search_area,
        {"color": "yellow"},
        "Pamba search extent"
    )

    candidate_map.addLayer(
        candidates9.style(
            color="red",
            fillColor="00000000",
            width=2
        ),
        {},
        "HydroBASINS level-9 candidates"
    )

    candidate_map

Level-9 candidate count: 46


,HYBAS_ID,MAIN_BAS,NEXT_DOWN,SUB_AREA,UP_AREA,area_km2,centroid_lonlat
0,4091153890,4090029720,4090029720,818.8,1136.2,818.836665,"[76.78344037610735, 9.211637118777443]"
1,4091152380,4090029250,4091152220,805.3,805.4,805.344999,"[77.47847524311155, 9.758333620258549]"
2,4091154180,4090029360,4091154020,804.0,804.0,803.951396,"[77.46057381355726, 9.253734890693753]"
3,4091153810,4090029720,4091153850,802.4,802.4,802.390623,"[76.7923221031366, 9.508034979424862]"
4,4091153780,4090029720,4091153910,783.0,783.2,783.027788,"[77.08890564283213, 9.39448318934221]"
5,4091152880,4090029810,4091152590,773.6,1183.0,773.588317,"[77.08316712951489, 9.633172363306883]"
6,4090029780,4090029780,0,660.9,660.9,660.942064,"[76.34376692539803, 9.973193679121527]"
7,4091152520,4090029250,4091152370,648.2,648.2,648.176217,"[77.30914082045051, 9.736567976237342]"
8,4090029660,4090029660,0,632.5,632.5,632.477632,"[76.82872404833807, 8.889605404224639]"
9,4090029680,4090029680,0,631.6,1507.1,631.606424,"[76.75927773158004, 9.046376483485098]"


## Step 2.31 — Inspect the candidate drainage group

**Objective:** Examine catchments associated with HydroBASINS group `4090029720` and compare their combined spatial pattern with the official Pamba basin map.

**Method:** Highlight the group on the map and inspect its individual catchments. The group is a candidate for investigation, not an accepted Pamba boundary.

**Decision rule:** Do not accept the group unless its shape, river connections, and basin definition are consistent with the official reference.

In [29]:
# Candidate group identified in the previous table.
target_main_basin = 4090029720

# Select catchments associated with this drainage group.
pamba_group = candidates9.filter(
    ee.Filter.eq("MAIN_BAS", target_main_basin)
)

# Identify the individual feature with this HYBAS_ID, if present.
group_root = candidates9.filter(
    ee.Filter.eq("HYBAS_ID", target_main_basin)
)

print("Catchments in candidate group:", pamba_group.size().getInfo())
print("Feature matching group ID:", group_root.size().getInfo())

# Summarize the group's catchments.
group_info = pamba_group.select([
    "HYBAS_ID",
    "NEXT_DOWN",
    "MAIN_BAS",
    "SUB_AREA",
    "UP_AREA"
]).getInfo()

group_rows = [
    feature["properties"]
    for feature in group_info["features"]
]

group_df = pd.DataFrame(group_rows)

if not group_df.empty:
    display(
        group_df.sort_values("SUB_AREA", ascending=False)
        .reset_index(drop=True)
    )

# Map the group against the wider candidate network.
group_map = geemap.Map(basemap="Esri.WorldTopoMap")
group_map.centerObject(pamba_search_area, 9)

group_map.addLayer(
    pamba_search_area,
    {"color": "yellow"},
    "Search extent"
)

group_map.addLayer(
    candidates9.style(
        color="777777",
        fillColor="00000000",
        width=1
    ),
    {},
    "All level-9 candidates"
)

group_map.addLayer(
    pamba_group.style(
        color="red",
        fillColor="FF000020",
        width=2
    ),
    {},
    "Candidate group 4090029720"
)

group_map.addLayer(
    group_root.style(
        color="orange",
        fillColor="FFA50030",
        width=3
    ),
    {},
    "Feature matching group ID"
)

group_map

Catchments in candidate group: 11
Feature matching group ID: 1


,HYBAS_ID,MAIN_BAS,NEXT_DOWN,SUB_AREA,UP_AREA
0,4091153890,4090029720,4090029720,818.8,1136.2
1,4091153810,4090029720,4091153850,802.4,802.4
2,4091153780,4090029720,4091153910,783.0,783.2
3,4090029720,4090029720,0,320.9,3955.4
4,4091153820,4090029720,4091153910,314.2,314.3
5,4091153830,4090029720,4091153850,299.3,1680.2
6,4091153960,4090029720,4091153830,274.3,274.3
7,4091154580,4090029720,4091153890,191.8,191.8
8,4091154590,4090029720,4091153890,125.2,125.3
9,4091153850,4090029720,4090029720,16.3,2498.7


Map(center=[9.425421146058019, 76.80000000000004], controls=(WidgetControl(options=['position', 'transparent_b…